In [ ]:
import requests
from pathlib import Path
import MDAnalysis as mda
from MDAnalysis.lib.util import NamedStream
from io import StringIO
import gemmi
import pymol
from glob import glob

# WARNING

Some PDB cleaning needs to be done by hand. This is case based. While the below functions might apply all necessary filters to most PDB files, it is important to check your final files visually. For example, CYP3A4 6daj has a floating GLN group that, after download and processing with this code, had to still be removed by hand. 

In [ ]:
uniprot_ids = {
               "CYP2D6": "P10635", 
               "CYP3A4": "P08684",
               "CYP1A2": "P05177",
               "CYP2C9": "P11712", 
               "PXR"   : "O75469", 
               "AHR"   : "P35869"
              }


In [ ]:
# User defined; Remove GLN from cyp3a4
common_co_crystals = ["ZN", "NA", "MG", "K", "ACT", "ER3", "SO4", "PO4", "NI", "MOO", #ions
                     "MPD", "IPA", "IMD", "EDO", "DMS", "GOL", "CIT", "PG0", #solvents
                     "2CV", "CPS", #detergents
                      "ADP"] #biological, in AHR

exclude_co_crystals = " ".join([f"and not resname {c}" for c in common_co_crystals ])

In [ ]:
#Code below will only do one ID at a time. Set interested KEY below.

target = "AHR"

#### **If you already have input data to process, skip to proper section. 
Not all sections should be run. 

Run cells for your needs. 

The following options are available:

- Download and save raw pdb files from RCSB and save processed final files
- Download raw pdb text from RCSB and save processed final files
- Have existing pdb files that you are interested in processing, and save final files

## Query PDB IDs

Get PDB IDs from UNIPROT ID using RCSB PDB search API

In [ ]:
def get_pdb_ids(uniprot_id, rows=1000):
    url = "https://search.rcsb.org/rcsbsearch/v2/query?json="
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": rows  # default 10, change if 1000 ids found 
            }
        }
    }
    response = requests.post(url, json=query)
    response.raise_for_status()
    result = response.json()
    pdb_ids = [entry["identifier"] for entry in result["result_set"]]
    if len(pdb_ids) < rows:
        print(f"Found {len(pdb_ids)} PDB IDs.")
    else:
        print(f"Found {len(pdb_ids)} PDB IDs. Consider changing rows to greater than {rows}.")
    return pdb_ids

In [ ]:
pdb_ids = get_pdb_ids(uniprot_ids[target])

If using IDs, this downloads PDBs using the RCSB API

In [ ]:
def get_rcsb_url(pdb_id, fmt="pdb"):
    url = f"https://files.rcsb.org/download/{pdb_id}.{fmt}"
    return requests.get(url)

def write_file(text, file_path):
    with open(file_path, "w") as f:
        f.write(text)

def convert_cif_to_pdb_gemmi(cif_text):
    """Convert mmCIF text to PDB string using gemmi."""
    doc = gemmi.cif.read_string(cif_text)
    structure = gemmi.make_structure_from_block(doc.sole_block())
    return structure.make_pdb_string()


In [ ]:
def get_rcsb_data_entry(pdb_id):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    r = requests.get(url)
    if r.status_code != 200:
        print(f"Failed to receive data entry for {pdb_id}")
        return []
    return r.json()

def get_ligand_chain_ids(entry, pdb_id):
    ligand_ids = entry.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", [])
    ligand_chains = {}
    for lig_id in ligand_ids:
        url = f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{lig_id}"
        r2 = requests.get(url)
        if r2.status_code != 200:
            continue
        data = r2.json()
        # print(data)
        chem_id = data['pdbx_entity_nonpoly']["comp_id"]
        chains = data["rcsb_nonpolymer_entity_container_identifiers"]["auth_asym_ids"]
        ligand_chains[chem_id] = chains   
    return ligand_chains


def get_chain_ids(entry, pdb_id, uniprot):
    entities = entry.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    chains = []
    for ent in entities:
        r2 = requests.get(
            f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{ent}"
        )
        if r2.status_code != 200:
            continue
        poly = r2.json()
        for ref in poly.get("rcsb_polymer_entity_container_identifiers", {}).get("reference_sequence_identifiers", []):
            if ref.get("database_accession") == uniprot:
                chains.extend(
                    poly.get("rcsb_polymer_entity_container_identifiers", {}).get("auth_asym_ids", [])
                )
    return sorted(set(chains))

In [ ]:
def get_rcsb_pdb(pdb_id, outdir=".", download_initial=False):
    """
    Download a PDB or mmCIF for the given PDB ID.
    Returns the final PDB content as a string. 
    The initial data file is saved, if specified. 
    """
    if download_initial:
        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)
        pdb_path = outdir / f"{pdb_id}_initial.pdb"

    # First try direct PDB download
    pdb_response = get_rcsb_url(pdb_id, fmt="pdb")
    if pdb_response.status_code == 200:
        print(f"Downloading {pdb_id} from RCSB...")
        pdb_text = pdb_response.text
        if download_initial:
            print(f"Saving {pdb_id} from RCSB...")
            write_file(pdb_text, pdb_path)
        return pdb_text

    # Fallback to CIF
    print(f"PDB for {pdb_id} not found. Checking for mmCIF...")
    cif_response = get_rcsb_url(pdb_id, fmt="cif")
    if cif_response.status_code == 200:
        print(f"mmCIF found. Converting to PDB...")
        pdb_text = convert_cif_to_pdb_gemmi(cif_response.text)
        if download_initial:
            write_file(pdb_text, pdb_path)
        return pdb_text

    raise ValueError(f"Neither PDB nor CIF available for {pdb_id}.")

def get_pdb_path(pdb_dir, pdb_id):
    return glob(f"{pdb_dir}/{pdb_id}*.pdb")[0]



In [ ]:
def process_pdb(pdb_id, 
                pdb_text="", 
                outdir=".", 
                exclude_resnames="", 
                chain_ids=None,
                input_path=None):
    """
    Process the PDB text to remove common co-crystals and
    save the final processed PDB.
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    final_path = outdir / f"{pdb_id}.pdb"

    if input_path:
        pdb = get_pdb_path(input_path, pdb_id)
        u = mda.Universe(pdb)
    else:
        u = mda.Universe(NamedStream(StringIO(pdb_text), f"{pdb_id}.pdb"))
        
    protein_A = u.select_atoms(f"protein and chainid {chain_ids[0]}")
    others = u.select_atoms(f"chainid {chain_ids[0]} and not protein and not water {exclude_resnames}")
    #Currently removing HEM dependency as it is only necessary in CYP, and doesn't include occurence of HEC
    # if 'HEM' not in set(others.resnames):
    #     print(f"Skipping {pdb_id}: HEM group not found.")
    #     return None
        
    #assert 'HEM' in set(others.resnames), "HEM group not found in structure."

    combined = protein_A + others
    lig = combined.select_atoms(f"chainid {chain_ids[0]} and not protein and not resname HEM and not resname HEC")
    
    if len(lig) > 0:
        lig.residues.resnames = ["LIG"] * len(lig.residues)
        #combined = combined + lig

    final_lig_set = set(combined.select_atoms(f"chainid {chain_ids[0]} and not protein and not water").resnames)
    #assert final_lig_set == {"HEM"} or final_lig_set == {"LIG", "HEM"}

    combined.write(final_path)
    return final_path

## Download Initial Files from RCSB and Save Processed Files

In [ ]:
input_dir = f"{target}/raw_pdb"
final_dir = f"{target}/final"

In [ ]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    print(f"Checking Chain IDs...")
    entry = get_rcsb_data_entry(_id)
    chain_ids = get_chain_ids(entry, _id, uniprot_ids[target])
    if 'A' not in chain_ids:
        print(f"Structure {_id} is selecting other than Chain A.")
    pdb_text = get_rcsb_pdb(_id, outdir=input_dir, download_initial=True)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_resnames=exclude_co_crystals, chain_ids=chain_ids)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")

## Only Grab RCSB PDB text and Save Processed Files

In [ ]:
final_dir = f"{target}/final"

In [ ]:
for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    print(f"Checking Chain IDs...")
    entry = get_rcsb_data_entry(_id)
    chain_ids = get_chain_ids(entry, _id, uniprot_ids[target])
    if 'A' not in chain_ids:
        print(f"Structure {_id} is selecting other than Chain A.")
    pdb_text = get_rcsb_pdb(_id)
    try:
        processed_path = process_pdb(_id, pdb_text, outdir=final_dir, exclude_resnames=exclude_co_crystals, chain_ids=chain_ids)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")

## Process from existing PDB

In [ ]:
#Set target to target of choice, so that for loop will loop over correct pdb_ids

pdb_dir_path = 'PATH/TO/PDB/FILES'
final_dir = 'PATH/TO/PROCESSED'

for _id in pdb_ids:
    _id = _id.lower()
    print(f"\nProcessing: {_id}")
    print(f"Checking Chain IDs...")
    entry = get_rcsb_data_entry(_id)
    chain_ids = get_chain_ids(entry, _id, uniprot_ids[target])
    if 'A' not in chain_ids:
        print(f"Structure {_id} is selecting other than Chain A.")
    try:
        processed_path = process_pdb(_id, outdir=final_dir, exclude_resnames=exclude_co_crystals, input_path=pdb_dir_path, chain_ids=chain_ids)
        print(f"Saved processed PDB: {processed_path}")
    except Exception as e:
        print(f"Failed to process {_id}: {e}")